In [ ]:
# Deriving weights for clustering temperature zones based on predictor -> predictand relationship

# V1 - 2026-08-17 - for final run - JPK

# Workflow

In [2]:
# libraries
import os
import numpy as np
import pandas as pd
import rasterio
import rioxarray as rxr
import xarray as xr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import gc
from pathlib import Path

# paths
sample_raster = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")
clim_file_path = r"..\Data_Temp\Processed\Spatial_clustering\Climatology_files"
clim_file_list = [f for f in os.listdir(clim_file_path) if f.endswith(".tif")]

# configurations
random_state_num = 64
n_cores = 6
target_var_tmax = 'tmax_climatology'
target_var_tmin = 'tmin_climatology'

In [3]:
# Stack the rasters 
data_arrays = {}
for file in clim_file_list:
    var_name = file.split(".")[0]
    da = rxr.open_rasterio(os.path.join(clim_file_path, file), masked=True)
    da = da.rio.reproject_match(sample_raster)
    
    if da.shape[0] == 1:
        da = da.squeeze('band', drop=True)
    
    data_arrays[var_name] = da

stacked_ds = xr.Dataset(data_arrays)
del data_arrays, sample_raster
gc.collect()

y_dim = stacked_ds.rio.y_dim  
x_dim = stacked_ds.rio.x_dim

valid_mask = None
for var in stacked_ds.data_vars:
    var_mask = ~np.isnan(stacked_ds[var].values)
    # add a nan filter for values = -9999
    valid_mask = var_mask if valid_mask is None else valid_mask & var_mask

stacked_ds_dict = {}
for var in stacked_ds.data_vars:
    stacked_ds_dict[var] = stacked_ds[var].values[valid_mask].flatten().astype('float32')

stacked_df = pd.DataFrame(stacked_ds_dict, dtype='float32')
del stacked_ds_dict, var_mask
gc.collect()

# Store spatial info
spatial_coords = {
    'x': stacked_ds[x_dim].values,
    'y': stacked_ds[y_dim].values,
    'crs': stacked_ds.rio.crs,
    'transform': stacked_ds.rio.transform(),
    'shape': (len(stacked_ds[y_dim]), len(stacked_ds[x_dim]))
}

del stacked_ds, valid_mask, y_dim, x_dim
gc.collect()

473

In [4]:
#3. Perform a simple random split

train_df, test_df = train_test_split(stacked_df, test_size=0.3, random_state=random_state_num)

X_train = train_df.drop(columns=[target_var_tmax, target_var_tmin])
y_train_tmax = train_df[target_var_tmax]
y_train_tmin = train_df[target_var_tmin]

X_test = test_df.drop(columns=[target_var_tmax, target_var_tmin])
y_test_tmax = test_df[target_var_tmax]
y_test_tmin = test_df[target_var_tmin]

feature_names = X_train.columns.tolist()

del stacked_df, train_df, test_df
gc.collect()

0

In [5]:
# 4. Fit random forest model


param_dist_rf = {
    'n_estimators': [100, 200, 300, 500], 
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30], 
    'min_samples_split': [2, 5, 10], 
    'min_samples_leaf': [1, 2, 4] 
}

rf = RandomForestRegressor(random_state=42)
rf_search = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=param_dist_rf, 
    n_iter=50,  
    cv=5, 
    verbose=3, 
    random_state=random_state_num, 
    n_jobs= n_cores,
    scoring='neg_root_mean_squared_error'  
)


In [ ]:
# Fit model and calculate Permutation Feature Importance and convert to relative weights
for t_var in ['tmax','tmin']:
#for t_var in ['tmin']:
    if t_var == 'tmax':
        y_train = y_train_tmax
        y_test = y_test_tmax
    else:
        y_train = y_train_tmin
        y_test = y_test_tmin    
    
    rf_search.fit(X_train, y_train)
    best_rf_model = rf_search.best_estimator_

    # 4. Final Validation Scores
    print(f"\nBest RF Hyperparameters for {t_var}: {rf_search.best_params_}")
    print(f"Best Mean CV Score (Train) {t_var}: {rf_search.best_score_:.4f}")
    print(f"Independent Test RMSE Score {t_var}: {best_rf_model.score(X_test, y_test):.4f}")

    pfi_test = permutation_importance(best_rf_model, X_test, y_test, scoring = 'neg_mean_absolute_error', random_state = random_state_num, n_repeats = 10, n_jobs = 1, max_samples = 0.25)

    feature_importances_abs = dict(zip(feature_names, pfi_test.importances_mean))

    #calculate relative importance 
    feature_importances = {k: v / sum(feature_importances_abs.values()) for k, v in feature_importances_abs.items()}

    #sort the dict by values in descending order
    feature_importances = dict(sorted(feature_importances.items(), key=lambda item: item[1], reverse=True))
    feature_importances_df = pd.DataFrame.from_dict(feature_importances, orient='index', columns=['importance'])
    feature_importances_df.to_csv(fr'..\Data_Temp\Processed\Spatial_clustering\climatology_weight_{t_var}.csv')
    print(feature_importances)


Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best RF Hyperparameters for tmin: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Best Mean CV Score (Train) tmin: -0.3230
Independent Test RMSE Score tmin: 0.9987
{'MODIS_lst_n_climatology': np.float64(0.15428314229466025), 'DEM_1km_elevation_mean': np.float64(0.10706862485634754), 'DEM_1km_elevation_max': np.float64(0.10594205863467264), 'GLDAS_wind_speed_climatology': np.float64(0.0878101849704919), 'DEM_1km_elevation_min': np.float64(0.08200995712208697), 'GLDAS_air_temp_climatology': np.float64(0.07991684355246279), 'GLDAS_lwdown_climatology': np.float64(0.0772708803707024), 'GLDAS_surface_pressure_climatology': np.float64(0.07171917093609848), 'GLDAS_specific_humidity_climatology': np.float64(0.06548544596711109), 'dem_smoothed': np.float64(0.0515606769465144), 'GLDAS_swdown_climatology': np.float64(0.035646925462049624), 'fetch_dist_1000m_resampled': np.